In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "nutrivision_yolov8s.keras"
LABEL_PATH = PROJECT_ROOT / "data" / "raw" / "FoodSeg103" / "id2label.json"
VAL_IMAGES = PROJECT_ROOT / "data" / "processed" / "foodseg103_yolo" / "images" / "val"
VAL_LABELS = PROJECT_ROOT / "data" / "processed" / "foodseg103_yolo" / "labels" / "val"

print("Project root:", PROJECT_ROOT)
print("Model exists:", MODEL_PATH.exists())
print("Labels exist:", LABEL_PATH.exists())
print("Validation images exist:", VAL_IMAGES.exists())
print("Validation labels exist:", VAL_LABELS.exists())

Project root: c:\Users\SANDEEP\Documents\NutriVision
Model exists: True
Labels exist: True
Validation images exist: True
Validation labels exist: True


In [2]:
import keras

model = keras.models.load_model(
    MODEL_PATH,
    compile=False,
)

print("Model type:", type(model))
print("Model loaded successfully.")

Model type: <class 'keras_cv.src.models.object_detection.yolo_v8.yolo_v8_detector.YOLOV8Detector'>
Model loaded successfully.


In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from PIL import Image
from src.detection.inference import predict

# Use one real validation image
image_path = sorted(VAL_IMAGES.glob("*.jpg"))[0]
image = Image.open(image_path)

result = predict(image, model)

print("Image:", image_path.name)
print("Original size:", result["original_size"])
print("Detections:", result["num_detections"])

for i, detection in enumerate(result["detections"], start=1):
    print(f"\nDetection {i}")
    print("  Label:", detection["label"])
    print("  FoodSeg103 ID:", detection["foodseg103_class_id"])
    print("  Confidence:", detection["confidence"])
    print("  Box:", detection["box"])

Image: 00000.jpg
Original size: (512, 485)
Detections: 0


In [5]:
image_paths = sorted(VAL_IMAGES.glob("*.jpg"))[:20]

total_detections = 0

for image_path in image_paths:
    image = Image.open(image_path)
    result = predict(image, model)

    print(
        image_path.name,
        "->",
        result["num_detections"],
        "detections",
        [
            (d["label"], round(d["confidence"], 3))
            for d in result["detections"]
        ],
    )

    total_detections += result["num_detections"]

print("\nTotal detections:", total_detections)

00000.jpg -> 0 detections []
00001.jpg -> 0 detections []
00002.jpg -> 0 detections []
00003.jpg -> 0 detections []
00004.jpg -> 1 detections [('sauce', 0.26)]
00005.jpg -> 0 detections []
00006.jpg -> 0 detections []
00007.jpg -> 0 detections []
00008.jpg -> 0 detections []
00009.jpg -> 0 detections []
00010.jpg -> 0 detections []
00011.jpg -> 0 detections []
00012.jpg -> 1 detections [('broccoli', 0.217)]
00013.jpg -> 0 detections []
00014.jpg -> 0 detections []
00015.jpg -> 0 detections []
00016.jpg -> 0 detections []
00017.jpg -> 1 detections [('milkshake', 0.32)]
00018.jpg -> 0 detections []
00019.jpg -> 0 detections []

Total detections: 3


In [6]:
from pathlib import Path

for image_path in image_paths:
    label_path = VAL_LABELS / f"{image_path.stem}.txt"

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    print(
        image_path.name,
        "->",
        len(lines),
        "ground-truth boxes"
    )

00000.jpg -> 5 ground-truth boxes
00001.jpg -> 3 ground-truth boxes
00002.jpg -> 5 ground-truth boxes
00003.jpg -> 3 ground-truth boxes
00004.jpg -> 3 ground-truth boxes
00005.jpg -> 2 ground-truth boxes
00006.jpg -> 6 ground-truth boxes
00007.jpg -> 5 ground-truth boxes
00008.jpg -> 2 ground-truth boxes
00009.jpg -> 6 ground-truth boxes
00010.jpg -> 3 ground-truth boxes
00011.jpg -> 4 ground-truth boxes
00012.jpg -> 4 ground-truth boxes
00013.jpg -> 2 ground-truth boxes
00014.jpg -> 5 ground-truth boxes
00015.jpg -> 3 ground-truth boxes
00016.jpg -> 6 ground-truth boxes
00017.jpg -> 3 ground-truth boxes
00018.jpg -> 4 ground-truth boxes
00019.jpg -> 5 ground-truth boxes


In [7]:
import json

with open(LABEL_PATH, "r", encoding="utf-8") as f:
    raw_labels = json.load(f)

labels = {int(k): v.strip() for k, v in raw_labels.items()}

for image_path in image_paths:
    label_path = VAL_LABELS / f"{image_path.stem}.txt"

    # Ground truth
    gt_classes = []
    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                class_id = int(float(line.split()[0]))
                foodseg_id = class_id + 1
                gt_classes.append(labels[foodseg_id])

    # Prediction
    image = Image.open(image_path)
    result = predict(image, model)

    pred_classes = [
        d["label"]
        for d in result["detections"]
    ]

    print(f"\n{image_path.name}")
    print("  GT :", gt_classes)
    print("  Pred:", pred_classes)


00000.jpg
  GT : ['dried cranberries', 'shrimp', 'rice', 'carrot', 'French beans']
  Pred: []

00001.jpg
  GT : ['shrimp', 'carrot', 'green beans']
  Pred: []

00002.jpg
  GT : ['olives', 'pineapple', 'sauce', 'bread', 'tomato']
  Pred: []

00003.jpg
  GT : ['chocolate', 'banana', 'grape']
  Pred: []

00004.jpg
  GT : ['sauce', 'fish', 'bread']
  Pred: ['sauce']

00005.jpg
  GT : ['chicken duck', 'rice']
  Pred: []

00006.jpg
  GT : ['biscuit', 'cheese butter', 'apple', 'grape', 'bread', 'other ingredients']
  Pred: []

00007.jpg
  GT : ['banana', 'strawberry', 'blueberry', 'kiwi', 'bread']
  Pred: []

00008.jpg
  GT : ['chicken duck', 'carrot']
  Pred: []

00009.jpg
  GT : ['steak', 'cucumber', 'cilantro mint', 'onion', 'shiitake', 'other ingredients']
  Pred: []

00010.jpg
  GT : ['chicken duck', 'pie', 'potato']
  Pred: []

00011.jpg
  GT : ['wine', 'sausage', 'pie', 'cilantro mint']
  Pred: []

00012.jpg
  GT : ['fish', 'corn', 'broccoli', 'green beans']
  Pred: ['broccoli']

0001

In [8]:
TRAIN_IMAGES = PROJECT_ROOT / "data" / "processed" / "foodseg103_yolo" / "images" / "train"
TRAIN_LABELS = PROJECT_ROOT / "data" / "processed" / "foodseg103_yolo" / "labels" / "train"

train_image_paths = sorted(TRAIN_IMAGES.glob("*.jpg"))[:20]

total_gt = 0
total_pred = 0

for image_path in train_image_paths:
    label_path = TRAIN_LABELS / f"{image_path.stem}.txt"

    gt_classes = []

    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                class_id = int(float(line.split()[0]))
                foodseg_id = class_id + 1
                gt_classes.append(labels[foodseg_id])

    image = Image.open(image_path)
    result = predict(image, model)

    pred_classes = [
        d["label"]
        for d in result["detections"]
    ]

    total_gt += len(gt_classes)
    total_pred += len(pred_classes)

    print(f"\n{image_path.name}")
    print("  GT  :", gt_classes)
    print("  Pred:", pred_classes)

print("\nTotal GT boxes:", total_gt)
print("Total predictions:", total_pred)


00000.jpg
  GT  : ['chicken duck', 'rice', 'snow peas']
  Pred: ['broccoli']

00001.jpg
  GT  : ['sauce', 'potato', 'cucumber', 'onion']
  Pred: []

00002.jpg
  GT  : ['bread', 'pie']
  Pred: []

00003.jpg
  GT  : ['chicken duck', 'carrot', 'asparagus', 'broccoli']
  Pred: ['broccoli']

00004.jpg
  GT  : ['steak', 'carrot', 'broccoli', 'French beans', 'white button mushroom']
  Pred: ['broccoli']

00005.jpg
  GT  : ['chicken duck', 'tomato', 'cilantro mint']
  Pred: ['broccoli']

00006.jpg
  GT  : ['sausage', 'sauce', 'corn', 'potato', 'carrot']
  Pred: []

00007.jpg
  GT  : ['sausage', 'rice', 'potato']
  Pred: ['rice']

00008.jpg
  GT  : ['cheese butter', 'egg', 'strawberry', 'bread']
  Pred: []

00009.jpg
  GT  : ['pork', 'carrot', 'green beans']
  Pred: []

00010.jpg
  GT  : ['biscuit', 'cheese butter']
  Pred: []

00011.jpg
  GT  : ['french fries', 'ice cream']
  Pred: []

00012.jpg
  GT  : ['ice cream', 'tomato']
  Pred: []

00013.jpg
  GT  : ['steak', 'sauce', 'bread', 'tomato'